In [40]:
import requests
import pandas as pd
import sqlite3
import random

### EXTRACT DATA FROM API 

In [41]:
try:
    # RANDOM NUMBERS TO SIMULATE REAL DATA
    rand_limit = random.randint(1, 100)
    rand_offset = random.randint(0, 1351-rand_limit)
    print('limit:', rand_limit, 'offset:', rand_offset)

    url = 'https://pokeapi.co/api/v2/pokemon/'
    param = {'limit':rand_limit,'offset':rand_offset }
    response = requests.get(url, params=param, timeout=15)
    response.raise_for_status()
    
    pokemon_list = response.json()
except:
    print('Request timed out')

limit: 37 offset: 261


Extract each name from the list and insert into dataframe

In [42]:
columns = ['id', 'name', 'base_experience', 'height', 'weight']

pokemon_rows = []

for item in pokemon_list['results']:
    url = item.get('url')    
    res = requests.get(url, timeout=5)
    res.raise_for_status()
    
    data = res.json()
    
    row = {key:data.get(key) for key in columns}
    pokemon_rows.append(row)
        
pokemon = pd.DataFrame(pokemon_rows, columns=columns)

In [43]:
pokemon

,id,name,base_experience,height,weight
0,262,mightyena,147,10,370
1,263,zigzagoon,56,4,175
2,264,linoone,147,5,325
3,265,wurmple,56,3,36
4,266,silcoon,72,6,100
5,267,beautifly,178,10,284
6,268,cascoon,72,7,115
7,269,dustox,173,12,316
8,270,lotad,44,5,26
9,271,lombre,119,12,325


### VALIDATE DATA

In [44]:
row_count = pokemon.shape[0]
print('Row Count:', row_count)
print('Equal to Limit:', row_count==rand_limit)

Row Count: 37
Equal to Limit: True


Set the proper data types.

In [45]:
pokemon[['id', 'base_experience', 'height', 'weight']] = (
    pokemon[['id', 'base_experience', 'height', 'weight']].astype('Int64')
    )

In [46]:
pokemon.dtypes

id                 Int64
name                 str
base_experience    Int64
height             Int64
weight             Int64
dtype: object

Check missing data

In [47]:
pokemon.isna().sum()

id                 0
name               0
base_experience    0
height             0
weight             0
dtype: int64

Check duplicates

In [48]:
dupe_count = pokemon.id.duplicated().sum()
if dupe_count == 0:
    print('No duplicates')
else:
    print('Duplicate IDs Found:', dupe_count)


No duplicates


Check for negative values

In [49]:
(pokemon.select_dtypes(include='number') < 0).any()

id                 False
base_experience    False
height             False
weight             False
dtype: boolean

### LOAD TO SQLITE DATABASE

Create db if not exists

In [50]:
conn = sqlite3.connect('data/pokedex.db')

conn.execute('''CREATE TABLE IF NOT EXISTS pokemons (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    base_experience INTEGER,
    height INTEGER NOT NULL,
    weight INTEGER NOT NULL
);''')

Check if IDs already in database

In [51]:
db_ids = pd.read_sql('select id from pokemons', conn)['id']
current_db_id_count = db_ids.count()
existing_ids = pokemon[pokemon['id'].isin(db_ids)]

new_ids = pokemon[~pokemon['id'].isin(db_ids)]

print('Existing IDs', existing_ids['id'].count())
print('New IDs', new_ids['id'].count())

Existing IDs 37
New IDs 0


Load pandas dataframe into sqlite db

In [52]:
new_ids.to_sql("pokemons", conn, if_exists='append', index=False)

0

In [53]:
pd.read_sql('select * from pokemons limit 5', conn)

,id,name,base_experience,height,weight
0,260,swampert,241,15,819
1,261,poochyena,56,5,136
2,262,mightyena,147,10,370
3,263,zigzagoon,56,4,175
4,264,linoone,147,5,325


In [54]:
inserted_count = len(new_ids)

print(inserted_count, 'new rows expected to be inserted')

db_row_count = conn.execute(
    'SELECT COUNT(*) FROM pokemons'
).fetchone()[0]

expected_db_count = current_db_id_count + inserted_count

if expected_db_count == db_row_count:
    print('Record counts match:', db_row_count, 'rows')
else:
    print(
        'Record count mismatch:',
        db_row_count, 'DB rows /',
        expected_db_count, 'expected rows'
    )

0 new rows expected to be inserted
Record counts match: 86 rows


Close db connection

In [55]:
conn.close()